# Predicción del precio de viviendas

Este cuaderno compara **Regresión Lineal** y **Árbol de Decisión** para estimar `price` con el dataset de Kaggle. Todas las líneas ejecutables incluyen un comentario personal que explica mi intención al usarlas.

## 1. Importar las librerías

Uso `pandas` para las tablas, `scikit-learn` para preparar y entrenar los modelos, y `matplotlib`/`seaborn` para interpretar visualmente el resultado.

In [ ]:
import numpy as np  # Yo uso NumPy para cálculos numéricos y para obtener raíces cuadradas.
import pandas as pd  # Yo uso pandas porque convierte el CSV en una tabla fácil de explorar.
import matplotlib.pyplot as plt  # Yo uso matplotlib para construir gráficas claras.
import seaborn as sns  # Yo uso seaborn para dar mejor presentación a las gráficas.
from pathlib import Path  # Yo uso Path para manejar rutas sin depender del sistema operativo.
from urllib.request import urlretrieve  # Yo uso esta función para descargar el archivo público desde Kaggle.
from zipfile import ZipFile  # Yo uso ZipFile porque Kaggle entrega este dataset comprimido.
from sklearn.compose import ColumnTransformer  # Yo uso este transformador para tratar columnas numéricas y de texto por separado.
from sklearn.impute import SimpleImputer  # Yo uso este imputador para que un valor faltante no detenga el modelo.
from sklearn.pipeline import Pipeline  # Yo uso Pipeline para asegurar que la preparación sea igual en entrenamiento y prueba.
from sklearn.preprocessing import OneHotEncoder  # Yo uso este codificador para convertir categorías en números que el modelo entiende.
from sklearn.model_selection import train_test_split  # Yo uso esta función para reservar datos nunca vistos para la evaluación.
from sklearn.linear_model import LinearRegression  # Yo importo el primer modelo: una relación lineal entre atributos y precio.
from sklearn.tree import DecisionTreeRegressor  # Yo importo el segundo modelo: reglas de decisión que pueden capturar no linealidades.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Yo importo tres métricas complementarias para juzgar el error.
sns.set_theme(style='whitegrid')  # Yo activo una cuadrícula suave porque facilita comparar valores en las gráficas.

## 2. Descargar y cargar los datos

La siguiente celda usa la descarga pública del dataset. Si Kaggle cambia la descarga directa, basta con subir manualmente `Housing.csv` a Colab y cambiar `ruta_csv`.

In [ ]:
url_dataset = 'https://www.kaggle.com/api/v1/datasets/download/yasserh/housing-prices-dataset'  # Yo guardo la fuente oficial para que el trabajo sea reproducible.
ruta_zip = Path('housing-prices-dataset.zip')  # Yo nombro el comprimido para identificar fácilmente qué descargué.
carpeta_datos = Path('datos_housing')  # Yo separo los datos descargados del resto del cuaderno.
urlretrieve(url_dataset, ruta_zip)  # Yo descargo el archivo comprimido una sola vez desde Kaggle.
with ZipFile(ruta_zip, 'r') as archivo_zip:  # Yo abro el ZIP en modo lectura para no modificar su contenido original.
    archivo_zip.extractall(carpeta_datos)  # Yo extraigo el CSV en una carpeta dedicada para poder leerlo después.
ruta_csv = carpeta_datos / 'Housing.csv'  # Yo indico el archivo concreto que contiene las viviendas.
viviendas = pd.read_csv(ruta_csv)  # Yo cargo el CSV en una tabla llamada viviendas.
print(f'Filas y columnas: {viviendas.shape}')  # Yo verifico el tamaño para confirmar que la lectura fue correcta.
display(viviendas.head())  # Yo miro las primeras filas porque me ayuda a reconocer las variables antes de modelar.
viviendas.info()  # Yo reviso tipos y valores no nulos para detectar problemas de calidad tempranamente.

## 3. Exploración inicial

Antes de entrenar, compruebo valores ausentes, observo el precio y separo el resultado que quiero predecir de los datos de entrada.

In [ ]:
print(viviendas.isna().sum())  # Yo cuento los faltantes por columna para no confundir ausencia de datos con un patrón real.
plt.figure(figsize=(9, 4))  # Yo creo un espacio ancho para que la distribución del precio se lea sin amontonarse.
sns.histplot(viviendas['price'], bins=30, kde=True, color='steelblue')  # Yo grafico el precio para reconocer su rango y posibles valores extremos.
plt.title('Distribución del precio de las viviendas')  # Yo titulo la gráfica para que su propósito sea evidente al revisar el informe.
plt.xlabel('Precio')  # Yo etiqueto el eje horizontal con la variable que estoy analizando.
plt.ylabel('Cantidad de viviendas')  # Yo aclaro que el eje vertical representa frecuencias.
plt.show()  # Yo muestro la gráfica antes de continuar con la partición de los datos.
X = viviendas.drop(columns='price')  # Yo retiro el precio de las entradas para evitar que el modelo vea la respuesta.
y = viviendas['price']  # Yo guardo el precio como la variable objetivo que deseo estimar.
columnas_numericas = X.select_dtypes(include=np.number).columns.tolist()  # Yo detecto números automáticamente para no clasificar columnas a mano.
columnas_categoricas = X.select_dtypes(exclude=np.number).columns.tolist()  # Yo reúno las categorías porque requieren una transformación distinta.
print('Variables numéricas:', columnas_numericas)  # Yo documento cuáles atributos quedan como cantidades.
print('Variables categóricas:', columnas_categoricas)  # Yo documento cuáles atributos se convertirán a indicadores.

## 4. Preparar datos y dividir la muestra

La codificación se ajusta únicamente con entrenamiento dentro de cada `Pipeline`; así se evita filtrar información del conjunto de prueba.

In [ ]:
transformador_numerico = Pipeline(steps=[('imputar', SimpleImputer(strategy='median'))])  # Yo reemplazo faltantes numéricos por la mediana, que resiste mejor valores extremos.
transformador_categorico = Pipeline(steps=[('imputar', SimpleImputer(strategy='most_frequent')), ('codificar', OneHotEncoder(handle_unknown='ignore'))])  # Yo completo categorías y las convierto en indicadores sin fallar ante una categoría nueva.
preprocesador = ColumnTransformer(transformers=[('numericas', transformador_numerico, columnas_numericas), ('categoricas', transformador_categorico, columnas_categoricas)])  # Yo reúno ambas transformaciones en una preparación única y ordenada.
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(X, y, test_size=0.20, random_state=42)  # Yo reservo el 20% y fijo una semilla para que el experimento se pueda repetir.
print(f'Entrenamiento: {X_entrenamiento.shape[0]} viviendas')  # Yo verifico cuántos casos aprende cada modelo.
print(f'Prueba: {X_prueba.shape[0]} viviendas')  # Yo verifico cuántos casos quedan para una evaluación honesta.

## 5. Entrenar Regresión Lineal y Árbol de Decisión

Limito la profundidad del árbol a 5 para reducir el riesgo de memorizar este conjunto pequeño. Es una configuración inicial, no una búsqueda exhaustiva de hiperparámetros.

In [ ]:
modelo_lineal = Pipeline(steps=[('preparar', preprocesador), ('modelo', LinearRegression())])  # Yo conecto la preparación y la regresión lineal para entrenarlas como un solo proceso.
modelo_arbol = Pipeline(steps=[('preparar', preprocesador), ('modelo', DecisionTreeRegressor(max_depth=5, min_samples_leaf=5, random_state=42))])  # Yo creo un árbol moderado para aprender reglas sin dividir grupos demasiado pequeños.
modelo_lineal.fit(X_entrenamiento, y_entrenamiento)  # Yo ajusto los coeficientes lineales usando únicamente los datos de entrenamiento.
modelo_arbol.fit(X_entrenamiento, y_entrenamiento)  # Yo ajusto las reglas del árbol usando exactamente el mismo conjunto de entrenamiento.
pred_lineal_entrenamiento = modelo_lineal.predict(X_entrenamiento)  # Yo calculo predicciones de entrenamiento para revisar posible sobreajuste lineal.
pred_lineal_prueba = modelo_lineal.predict(X_prueba)  # Yo calculo predicciones nuevas para medir la generalización lineal.
pred_arbol_entrenamiento = modelo_arbol.predict(X_entrenamiento)  # Yo calculo predicciones de entrenamiento del árbol para comparar su ajuste.
pred_arbol_prueba = modelo_arbol.predict(X_prueba)  # Yo calculo predicciones nuevas del árbol para hacer la comparación justa.

## 6. Medir y comparar el rendimiento

- **R²:** proporción de variación del precio explicada por el modelo; cuanto mayor, mejor.
- **MAE:** error medio absoluto en unidades de precio; cuanto menor, mejor.
- **RMSE:** penaliza más los errores grandes; cuanto menor, mejor.

In [ ]:
def calcular_metricas(y_real, y_estimado):  # Yo encapsulo las métricas para aplicar exactamente el mismo criterio a ambos modelos.
    return {'R2': r2_score(y_real, y_estimado), 'MAE': mean_absolute_error(y_real, y_estimado), 'RMSE': np.sqrt(mean_squared_error(y_real, y_estimado))}  # Yo devuelvo tres medidas para no depender de una sola perspectiva del error.
resultados = pd.DataFrame({'Regresión lineal - entrenamiento': calcular_metricas(y_entrenamiento, pred_lineal_entrenamiento), 'Regresión lineal - prueba': calcular_metricas(y_prueba, pred_lineal_prueba), 'Árbol de decisión - entrenamiento': calcular_metricas(y_entrenamiento, pred_arbol_entrenamiento), 'Árbol de decisión - prueba': calcular_metricas(y_prueba, pred_arbol_prueba)}).T  # Yo organizo todas las métricas en una tabla comparable por modelo y partición.
display(resultados.style.format({'R2': '{:.3f}', 'MAE': '{:,.0f}', 'RMSE': '{:,.0f}'}))  # Yo formateo la tabla para leer rápidamente precisión y error monetario.
resultados_prueba = resultados.loc[['Regresión lineal - prueba', 'Árbol de decisión - prueba']]  # Yo aíslo la prueba porque es la evidencia principal para escoger un modelo.
ax = resultados_prueba[['MAE', 'RMSE']].plot(kind='bar', figsize=(10, 5), color=['#4C78A8', '#F58518'])  # Yo comparo visualmente los errores de los dos modelos.
ax.set_title('Errores en datos de prueba: menor es mejor')  # Yo explico el criterio de lectura directamente en la gráfica.
ax.set_ylabel('Error en unidades de precio')  # Yo indico que las barras representan distancia respecto al precio real.
ax.set_xlabel('Modelo')  # Yo nombro el eje para que cada barra se pueda asociar a su modelo.
plt.xticks(rotation=0)  # Yo mantengo las etiquetas horizontales para hacerlas legibles.
plt.show()  # Yo presento la comparación antes de sacar una conclusión.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))  # Yo creo dos paneles para contrastar predicciones reales y estimadas lado a lado.
ejes[0].scatter(y_prueba, pred_lineal_prueba, alpha=0.7, color='#4C78A8')  # Yo dibujo cada predicción lineal frente a su valor real.
ejes[1].scatter(y_prueba, pred_arbol_prueba, alpha=0.7, color='#F58518')  # Yo dibujo cada predicción del árbol frente a su valor real.
limite_inferior = min(y_prueba.min(), pred_lineal_prueba.min(), pred_arbol_prueba.min())  # Yo hallo el menor valor para que ambas diagonales usen la misma escala.
limite_superior = max(y_prueba.max(), pred_lineal_prueba.max(), pred_arbol_prueba.max())  # Yo hallo el mayor valor para que ambas diagonales usen la misma escala.
for eje, titulo in zip(ejes, ['Regresión lineal', 'Árbol de decisión']):  # Yo recorro los paneles para aplicar el mismo criterio visual a los modelos.
    eje.plot([limite_inferior, limite_superior], [limite_inferior, limite_superior], 'k--')  # Yo añado la diagonal ideal donde predicción y realidad coinciden.
    eje.set_title(titulo)  # Yo identifico el modelo mostrado en cada panel.
    eje.set_xlabel('Precio real')  # Yo marco el precio observado en el eje horizontal.
    eje.set_ylabel('Precio predicho')  # Yo marco el precio calculado por el modelo en el eje vertical.
plt.tight_layout()  # Yo ajusto los espacios para que títulos y etiquetas no se superpongan.
plt.show()  # Yo muestro las gráficas que ayudan a detectar sesgos o dispersiones.

## 7. Análisis y conclusión

La celda siguiente construye una interpretación usando las métricas que realmente se obtengan. De esta forma, la conclusión no queda escrita antes de ver los resultados.

In [ ]:
mejor_modelo = resultados_prueba['RMSE'].idxmin()  # Yo elijo como ganador al modelo con menor penalización por errores grandes.
modelo_con_mayor_r2 = resultados_prueba['R2'].idxmax()  # Yo verifico por separado cuál explica mejor la variación del precio.
brecha_lineal = resultados.loc['Regresión lineal - entrenamiento', 'RMSE'] - resultados.loc['Regresión lineal - prueba', 'RMSE']  # Yo calculo la diferencia de error lineal entre aprender y generalizar.
brecha_arbol = resultados.loc['Árbol de decisión - entrenamiento', 'RMSE'] - resultados.loc['Árbol de decisión - prueba', 'RMSE']  # Yo calculo la misma diferencia para vigilar sobreajuste del árbol.
print(f'Conclusión principal: {mejor_modelo} obtuvo el menor RMSE de prueba, por lo que es la mejor opción en esta partición.')  # Yo resumo la decisión con la métrica más sensible a errores costosos.
print(f'El mayor R² de prueba pertenece a: {modelo_con_mayor_r2}.')  # Yo confirmo el modelo que explica una fracción mayor de la variación.
print(f'Brecha RMSE lineal (entrenamiento - prueba): {brecha_lineal:,.0f}.')  # Yo reporto si la regresión cambia mucho fuera de los datos que vio.
print(f'Brecha RMSE árbol (entrenamiento - prueba): {brecha_arbol:,.0f}.')  # Yo reporto si el árbol parece aprender demasiado los casos de entrenamiento.
print('Interpretación: si la brecha de un modelo es muy negativa, su error de prueba fue mayor y hay señal de sobreajuste; si es pequeña, su comportamiento es más estable.')  # Yo dejo una regla sencilla para interpretar las brechas sin ocultar el criterio.